# Сопоставление подписантов JSON с CSV
Поддерживаются форматы «Иванов А.А.», «А.А. Иванов» и полные ФИО. При несовпадении инициалов или сходстве фамилии ниже порога выводится «Совпадение не найдено». Измените настройки в начале следующей ячейки.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

# ===== НАСТРОЙКИ =====
JSON_DIR = Path("input_json")
CSV_FILE = "signers.csv"
OUTPUT_FILE = "result.xlsx"
JSON_KEYS = ["signer1", "signer2"]
CSV_COLUMN = "ФИО"
SURNAME_THRESHOLD = 0.75


def parse_fio(text):
    """Иванов А.А. и Иванов Андрей Александрович -> (иванов, аа)."""
    text = str(text).strip().lower().replace("ё", "е")
    parts = re.findall(r"[а-яa-z]+", text)
    if not parts:
        return "", ""
    return parts[0], "".join(part[0] for part in parts[1:3])


def parse_fio_auto(text):
    text = str(text).strip().lower().replace("ё", "е")
    parts = re.findall(r"[а-яa-z]+", text)
    if len(parts) >= 3 and len(parts[0]) == 1 and len(parts[1]) == 1:
        return parts[2], parts[0] + parts[1]
    return parse_fio(text)


def ngrams(text, n):
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i+n] for i in range(len(text)-n+1)}


def ngram_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    scores = []
    for n in (2, 3):
        x, y = ngrams(a, n), ngrams(b, n)
        if x and y:
            scores.append(len(x & y) / len(x | y))
    return sum(scores) / len(scores) if scores else 0


def initials_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    matches = sum(x == y for x, y in zip(a, b))
    return matches / max(len(a), len(b))


def similarity(fio1, fio2):
    surname1, initials1 = parse_fio(fio1)
    surname2, initials2 = parse_fio(fio2)
    if not surname1 or not surname2:
        return 0
    surname_score = ngram_similarity(surname1, surname2)
    initials_score = initials_similarity(initials1, initials2)
    if not initials1 or not initials2:
        return surname_score * 0.7
    score = surname_score * 0.7 + initials_score * 0.3
    if initials1 != initials2:
        score *= 0.7
    return score


def check_match(fio1, fio2):
    surname1, initials1 = parse_fio_auto(fio1)
    surname2, initials2 = parse_fio_auto(fio2)
    surname_score = ngram_similarity(surname1, surname2)
    valid = (
        len(initials1) == 2
        and len(initials2) == 2
        and initials1 == initials2
        and surname_score >= SURNAME_THRESHOLD
    )
    return valid, surname_score


df = pd.read_csv(CSV_FILE, sep=";", encoding="utf-8-sig")
signers = [name for name in df[CSV_COLUMN].dropna().astype(str) if name.strip()]
if not signers:
    raise ValueError("В указанном столбце CSV нет ФИО")

results = []
for file in JSON_DIR.glob("*.json"):
    with open(file, encoding="utf-8") as f:
        data = json.load(f)
    for key in JSON_KEYS:
        original = data.get(key)
        if not isinstance(original, str) or not original.strip():
            continue
        matches = [(signer, *check_match(original, signer)) for signer in signers]
        valid_matches = [match for match in matches if match[1]]
        if valid_matches:
            best_name, _, best_score = max(valid_matches, key=lambda x: x[2])
            status = "Совпадение найдено"
        else:
            best_name = ""
            best_score = 0
            status = "Совпадение не найдено"
        results.append({
            "Файл": file.name,
            "Ключ JSON": key,
            "Исходное ФИО": original,
            "Лучшее совпадение CSV": best_name,
            "Сходство, %": round(best_score * 100, 2),
            "Статус": status,
        })

columns = ["Файл", "Ключ JSON", "Исходное ФИО", "Лучшее совпадение CSV", "Сходство, %", "Статус"]
pd.DataFrame(results, columns=columns).to_excel(OUTPUT_FILE, index=False)
print(f"Готово! Обработано записей: {len(results)}")
print(f"Результат: {OUTPUT_FILE}")